# Test LLM — Description iconographique (Claude API)

**Projet** : Gallica Images — Illustrations des *Métamorphoses* d'Ovide  
**Date**   : Avril 2026

Ce notebook teste Claude (Anthropic) sur deux tâches :

1. **Description visuelle** — donner une illustration au LLM et lui demander
   de décrire ce qu'il voit, sans lui dire le thème.
2. **Connaissance iconographique** — demander au LLM de décrire les motifs
   canoniques du thème sans lui donner d'image.

**Thèmes testés** : Déluge (p.11) — Fin du Déluge (p.12) — Création du monde (p.3) — Création de l'homme (p.4)  
**Source** : Corpus Salomon 1557 — illustrations déjà segmentées par YOLO

---

## 1. Configuration

In [ ]:
import base64, requests, os, textwrap
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt

ANTHROPIC_API_KEY = "ta_cle_api"
MODELE_CLAUDE     = "claude-opus-4-5"

DOSSIER_SEG = "../data/segmentees/bois_salomon_lyon1557"

ILLUSTRATIONS = {
    "Déluge"              : 11,
    "Fin du Déluge"       : 12,
    "Création du monde"   : 3,
    "Création de l'homme" : 4,
}

print("✓ Configuration chargée")

## 2. Chargement des illustrations segmentées

In [ ]:
def charger_illustration(page):
    candidats = [
        f for f in os.listdir(DOSSIER_SEG)
        if f"_f{page:03d}_" in f and "_flip" not in f
    ]
    if not candidats:
        print(f"  ⚠️ Aucune illustration trouvée pour page {page}")
        return None, None
    chemin = f"{DOSSIER_SEG}/{sorted(candidats)[0]}"
    img    = Image.open(chemin).convert("RGB")
    buf    = BytesIO()
    img.save(buf, format="JPEG", quality=90)
    b64    = base64.b64encode(buf.getvalue()).decode()
    print(f"  ✓ Page {page} → {sorted(candidats)[0]}")
    return img, b64

illustrations_chargees = {}
for theme, page in ILLUSTRATIONS.items():
    img, b64 = charger_illustration(page)
    if img:
        illustrations_chargees[theme] = {"img": img, "b64": b64, "page": page}

# Aperçu
fig, axes = plt.subplots(1, len(illustrations_chargees),
                         figsize=(5 * len(illustrations_chargees), 5))
for ax, (theme, data) in zip(axes, illustrations_chargees.items()):
    ax.imshow(data["img"])
    ax.set_title(f"{theme}\npage {data['page']}", fontsize=9, fontweight="bold")
    ax.axis("off")
plt.suptitle("Illustrations à tester — Salomon 1557", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Fonctions Claude

In [ ]:
def appeler_claude_vision(b64, prompt):
    """Envoie une image + prompt à Claude et retourne la réponse."""
    r = requests.post(
        "https://api.anthropic.com/v1/messages",
        headers={
            "x-api-key"         : ANTHROPIC_API_KEY,
            "anthropic-version" : "2023-06-01",
            "content-type"      : "application/json"
        },
        json={
            "model"      : MODELE_CLAUDE,
            "max_tokens" : 800,
            "messages"   : [{
                "role"    : "user",
                "content" : [
                    {"type": "image",
                     "source": {"type": "base64",
                                "media_type": "image/jpeg",
                                "data": b64}},
                    {"type": "text", "text": prompt}
                ]
            }]
        },
        timeout=60
    )
    data = r.json()
    if r.status_code != 200:
        raise Exception(f"HTTP {r.status_code} — {data}")
    return data["content"][0]["text"]


def appeler_claude_texte(prompt):
    """Interroge Claude sans image."""
    r = requests.post(
        "https://api.anthropic.com/v1/messages",
        headers={
            "x-api-key"         : ANTHROPIC_API_KEY,
            "anthropic-version" : "2023-06-01",
            "content-type"      : "application/json"
        },
        json={
            "model"      : MODELE_CLAUDE,
            "max_tokens" : 800,
            "messages"   : [{"role": "user", "content": prompt}]
        },
        timeout=60
    )
    data = r.json()
    if r.status_code != 200:
        raise Exception(f"HTTP {r.status_code} — {data}")
    return data["content"][0]["text"]


print("✓ Fonctions Claude définies")

## 4. Prompts

In [ ]:
PROMPT_VISUEL = """Tu es un expert en iconographie de la Renaissance européenne.
Décris précisément cette illustration — personnages, actions, éléments du décor,
composition. Identifie le thème iconographique représenté et les motifs visuels
qui t'ont permis de l'identifier. Concentre-toi uniquement sur ce qui est
représenté, pas sur le style artistique ni la technique de gravure."""

PROMPTS_TEXTUELS = {
    "Déluge" :
        """Tu es un expert en iconographie de la Renaissance européenne.
Sans regarder d'image, décris les motifs visuels canoniques d'une illustration
du Déluge dans une édition illustrée des Métamorphoses d'Ovide du 16e siècle.
Attention : il s'agit du Déluge ovidien (Jupiter, Deucalion, Pyrrha) et non
du Déluge biblique (Noé). Quels éléments un graveur de cette époque
aurait-il représentés ? Réponds de manière précise et concise.""",

    "Création du monde" :
        """Tu es un expert en iconographie de la Renaissance européenne.
Sans regarder d'image, décris les motifs visuels canoniques d'une illustration
de la Création du monde dans une édition illustrée des Métamorphoses d'Ovide
du 16e siècle. Il s'agit de la Création ovidienne (le Chaos primordial, le
Démiurge, les quatre éléments) et non de la Création biblique.
Quels éléments un graveur de cette époque aurait-il représentés ?
Réponds de manière précise et concise."""
}

print("✓ Prompts définis")

## 5. Test 1 — Description visuelle

Chaque illustration est envoyée à Claude sans indication du thème.

In [ ]:
resultats_visuels = {}

for theme, data in illustrations_chargees.items():
    print(f"\n{'═'*60}")
    print(f"  {theme} — page {data['page']}")
    print(f"{'═'*60}")
    try:
        reponse = appeler_claude_vision(data["b64"], PROMPT_VISUEL)
        resultats_visuels[theme] = reponse
        print(reponse)
    except Exception as e:
        print(f"  Erreur : {e}")
        resultats_visuels[theme] = f"ERREUR : {e}"

## 6. Test 2 — Connaissance iconographique sans image

Claude décrit les motifs canoniques du Déluge et de la Création ovidiens sans voir d'illustration.

In [ ]:
resultats_textuels = {}

for theme, prompt in PROMPTS_TEXTUELS.items():
    print(f"\n{'═'*60}")
    print(f"  {theme} — sans image")
    print(f"{'═'*60}")
    try:
        reponse = appeler_claude_texte(prompt)
        resultats_textuels[theme] = reponse
        print(reponse)
    except Exception as e:
        print(f"  Erreur : {e}")
        resultats_textuels[theme] = f"ERREUR : {e}"

## 7. Visualisation — Illustration + description côte à côte

In [ ]:
for theme, data in illustrations_chargees.items():
    if theme not in resultats_visuels:
        continue

    fig, (ax_img, ax_txt) = plt.subplots(
        1, 2, figsize=(16, 7),
        gridspec_kw={"width_ratios": [1, 1.8]}
    )
    fig.patch.set_facecolor("#f8f5f0")
    fig.suptitle(
        f"{theme} — page {data['page']} — Description visuelle (Claude)",
        fontsize=12, fontweight="bold"
    )

    ax_img.imshow(data["img"])
    ax_img.set_title("Illustration — Salomon 1557", fontsize=9, fontweight="bold")
    ax_img.axis("off")

    ax_txt.axis("off")
    reponse = resultats_visuels[theme]
    texte   = "\n".join([textwrap.fill(l, width=70) for l in reponse.split("\n")])
    ax_txt.text(0.03, 0.97, texte,
                transform=ax_txt.transAxes,
                fontsize=8, va="top", ha="left",
                bbox=dict(boxstyle="round", facecolor="white",
                          alpha=0.9, edgecolor="#b71c1c"))
    ax_txt.set_title(MODELE_CLAUDE, fontsize=9,
                     fontweight="bold", color="#b71c1c")

    plt.tight_layout()
    plt.show()
    print("─" * 70)

## 8. Visualisation — Connaissance iconographique sans image

In [ ]:
for theme, reponse in resultats_textuels.items():
    fig, ax = plt.subplots(figsize=(14, 6))
    fig.patch.set_facecolor("#f8f5f0")
    fig.suptitle(
        f"{theme} — Connaissance iconographique sans image (Claude)",
        fontsize=12, fontweight="bold"
    )
    ax.axis("off")
    texte = "\n".join([textwrap.fill(l, width=90) for l in reponse.split("\n")])
    ax.text(0.03, 0.97, texte,
            transform=ax.transAxes,
            fontsize=8.5, va="top", ha="left",
            bbox=dict(boxstyle="round", facecolor="white",
                      alpha=0.9, edgecolor="#b71c1c"))
    ax.set_title(MODELE_CLAUDE, fontsize=9,
                 fontweight="bold", color="#b71c1c")
    plt.tight_layout()
    plt.show()
    print("─" * 70)